## 1. Tiền xử lý

In [1]:
from pathlib import Path
import sys

project = Path.cwd()
if project.name == "notebooks":
    project = project.parent
sys.path.insert(0, str(project))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from src.evaluation import DATA, RESULTS, SEED, load_data, make_folds, save_submission
from sklearn.base import clone
from sklearn.model_selection import GridSearchCV, cross_val_score
from sklearn.pipeline import Pipeline
from src.features import add_features, preprocess

X, y_log, X_test, test_ids, sample = load_data()
processor = preprocess(scale=False)

## 2. Tạo đặc trưng

In [2]:
feature_sets = {
    "basic": (X, X_test),
    "engineered": (add_features(X), add_features(X_test)),
}
folds = list(make_folds().split(X))


## 3. Huấn luyện và tuning 

In [3]:
from xgboost import XGBRegressor

name = "xgboost"
pipeline = Pipeline([("preprocess", processor), ("model", XGBRegressor(n_estimators=350, learning_rate=0.05, random_state=SEED, n_jobs=2, tree_method="hist"))])
grid = {"model__max_depth": [2, 3, 4]}
scoring = "neg_root_mean_squared_error"

# So sánh đặc trưng với cùng mô hình và tham số.
ablation = {
    label: -cross_val_score(pipeline, train_features, y_log, cv=folds, scoring=scoring)
    for label, (train_features, _) in feature_sets.items()
}
feature_choice = min(ablation, key=lambda label: ablation[label].mean())
train_features, test_features = feature_sets[feature_choice]
search = GridSearchCV(pipeline, grid, cv=folds, scoring=scoring, error_score="raise")
search.fit(train_features, y_log)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.","Pipeline(step...=None, ...))])"
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'model__max_depth': [2, 3, ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'neg_root_mean_squared_error'
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.","[(array([ 0, ...shape=(1168,)), ...), (array([ 0, ...shape=(1168,)), ...), ...]"
,"error_score error_score: 'raise' or numeric, default=np.nanValue to assign to the score if an error occurs in estimator fitting.If set to 'raise', the error is raised. If a numeric value is given,FitFailedWarning is raised. This parameter does not affect the refitstep, which will always raise the error.",'raise'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",None
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see h

In [4]:
for label, scores in ablation.items():
    print(f"{label}: CV RMSE = {scores.mean():.5f}")

print("Bộ đặc trưng đưa vào tinh chỉnh:", feature_choice)
print("Tham số tốt nhất:", search.best_params_)
print(f"CV RMSE sau tinh chỉnh: {-search.best_score_:.5f}")

basic: CV RMSE = 0.14224
engineered: CV RMSE = 0.13830
Bộ đặc trưng đưa vào tinh chỉnh: engineered
Tham số tốt nhất: {'model__max_depth': 3}
CV RMSE sau tinh chỉnh: 0.13211


**Nhận xét.** Trước khi tinh chỉnh, XGBoost đạt RMSE kiểm định chéo trung bình là **0,14224** với bộ đặc trưng cơ bản và **0,13830** với bộ đặc trưng bổ sung. Nhóm chọn bộ đặc trưng bổ sung để tinh chỉnh vì có sai số thấp hơn. Trong các cấu hình đã thử, `max_depth = 3` cho kết quả tốt nhất, với RMSE trung bình trên thang log giảm từ **0,13830** xuống **0,13211**.

## 4. Đánh giá và xuất CSV

In [5]:
import json

rows = []
for variant, scores in ablation.items():
    rows.append({"model": name, "variant": variant, "features": variant, "params": "default", **{
        f"fold_{i}": score for i, score in enumerate(scores, 1)
    }})
tuned_scores = [-search.cv_results_[f"split{i}_test_score"][search.best_index_] for i in range(len(folds))]
rows.append({"model": name, "variant": "tuned", "features": feature_choice,
             "params": json.dumps(search.best_params_), **{
                 f"fold_{i}": score for i, score in enumerate(tuned_scores, 1)
             }})
scores = pd.DataFrame(rows)
fold_columns = [f"fold_{i}" for i in range(1, len(folds) + 1)]
scores["cv_mean"] = scores[fold_columns].mean(axis=1)
scores["cv_std"] = scores[fold_columns].std(axis=1, ddof=0)
scores["selected"] = False
best = scores["cv_mean"].idxmin()
scores.loc[best, "selected"] = True
selected = scores.loc[best]
if selected["variant"] == "tuned":
    final_model = search.best_estimator_
else:
    train_features, test_features = feature_sets[selected["features"]]
    final_model = clone(pipeline).fit(train_features, y_log)
submission = save_submission(name, final_model.predict(test_features), test_ids, sample)
cv_dir = RESULTS / "cv"
cv_dir.mkdir(parents=True, exist_ok=True)
scores.to_csv(cv_dir / f"{name}.csv", index=False)
display(scores)
display(submission.head())


,model,variant,features,params,fold_1,fold_2,fold_3,fold_4,fold_5,cv_mean,cv_std,selected
0,xgboost,basic,basic,default,0.146685,0.135125,0.175182,0.136602,0.117622,0.142243,0.018939,False
1,xgboost,engineered,engineered,default,0.142173,0.131056,0.170333,0.132566,0.115359,0.138297,0.018181,False
2,xgboost,tuned,engineered,"{""model__max_depth"": 3}",0.132176,0.111792,0.168736,0.132241,0.115601,0.132109,0.020134,True


,Id,SalePrice
0,1461,122237.976562
1,1462,160989.703125
2,1463,188850.906250
3,1464,194142.859375
4,1465,192038.125000


### Nhận xét kết quả đánh giá và xuất CSV

Dựa trên kết quả đánh giá mô hình XGBoost bằng phương pháp kiểm định chéo 5-Fold, có thể đưa ra một số nhận xét như sau:

- Mô hình Basic: Đạt RMSE trung bình là 0.142243 và độ lệch chuẩn là 0.018939.
- Mô hình Engineered: Đạt RMSE trung bình là 0.138297 và độ lệch chuẩn là 0.018181. Kết quả có cải thiện so với mô hình Basic.
- Mô hình Tuned: Đạt RMSE trung bình thấp nhất là 0.132109, với độ lệch chuẩn là 0.020134. Đây là mô hình được lựa chọn để thực hiện dự đoán cuối cùng.

Nhìn chung, mô hình Tuned cho kết quả dự đoán tốt nhất trong ba phương án vì có RMSE trung bình thấp nhất. Tuy nhiên, độ lệch chuẩn cao hơn một chút so với hai mô hình còn lại, cho thấy kết quả giữa các Fold có sự biến động nhất định.

Sau khi lựa chọn mô hình tốt nhất, kết quả đánh giá được xuất thành file CSV để lưu trữ và phục vụ việc so sánh. Đồng thời, mô hình được sử dụng để dự đoán giá nhà trên tập dữ liệu kiểm tra và xuất kết quả thành file `submission.csv` với hai cột `Id` và `SalePrice`.